# Rainbow DQN 업그레이드 계획 (Seaquest)

> 이 노트북은 **현재 코드(`model.py`, `replay_buffer.py`, `train.py`, `main.py`, `env_utils.py`)에 변경을 적용하지 않은** 상태의 **작업 정리 문서**입니다.
> 아래 코드 셀은 모두 **참고용 스니펫**이며, 그대로 실행하도록 의도된 것이 아닙니다 (적용 시점에 별도 반영).

## 현재 상태 한눈에

| 영역 | 항목 | 상태 |
|---|---|---|
| Rainbow | Double DQN | ✅ |
| Rainbow | Dueling | ✅ |
| Rainbow | Prioritized Replay (PER) | ✅ |
| Rainbow | Multi-step (n-step) | ✅ |
| Rainbow | Noisy Nets | ✅ |
| Rainbow | **Distributional (C51)** | ❌ **미구현** |
| 최적화 | 리플레이 버퍼 메모리 (frame 공유) | ✅ 적용됨 (single frame ring buffer) |
| 최적화 | Mixed Precision (AMP) | ⏭️ 보류 (벡터화 env 이후 적용 예정) |
| 최적화 | 벡터화 환경(병렬 수집) | ⚠️ 단일 env |
| 최적화 | PER 샘플링 벡터화 | ✅ 적용됨 (numpy 벡터화) |
| 최적화 | NoisyNet 노이즈 디바이스 | ✅ 적용됨 (weight_mu.device 직접 생성) |
| 최적화 | target_net Noisy 정합성 | ✅ 적용됨 (eval() 제거) |
| 최적화 | non_blocking 전송 | ✅ 적용됨 (_gather .to(non_blocking=True)) |

**결론:** 알고리즘 완성도 5/6 (≈83%), C51만 추가하면 풀 Rainbow. 최적화는 AMP·벡터화 env 2건 미적용.

## 우선순위 로드맵

| 순위 | 업그레이드 | 효과 | 작업량 | 종류 |
|---|---|---|---|---|
| 1 | 리플레이 버퍼 메모리 최적화 | RAM ~1/8 → capacity 1M 가능 (학습 품질↑) | 중 | 최적화 |
| 2 | Mixed Precision (AMP) | 학습 속도 1.3~2배 | 소 | 최적화 |
| 3 | Distributional C51 | 풀 Rainbow 완성, 성능↑ | 대 | 알고리즘 |
| 4 | 벡터화 환경 | wall-clock throughput↑ | 대 | 구조 |
| 5 | PER 샘플링 벡터화 | train step 오버헤드↓ | 중 | 최적화 |
| 6 | NoisyNet 디바이스 / 기타 | 미세 속도↑, 정합성 | 소 | 정리 |

> 권장 순서: **1 → 2** 를 먼저 (투자 대비 효과 최고, 1번은 capacity를 키워 품질까지 개선) → 그다음 **3(C51)**.

---
## 이미 적용되어 있는 최적화 (의도 + 알게 모르게)

> 업그레이드와 별개로, **현재 코드에 이미 들어가 있는** 효율/안정화 요소 정리. 
> 🎯 = 표준 베스트프랙티스(의도적), 🍀 = 부수적으로 얻은 이득 / 표준 래퍼에 묻어온 것.

### 메모리
| | 내용 | 위치 | 효과 |
|---|---|---|---|
| 🎯 | obs를 **uint8** 저장 (float32 아님) | `replay_buffer.py:13-17` | RAM **4배** 절약 |
| 🎯 | 정규화 `/255.0` 를 **GPU forward 안**에서 수행 | `model.py:103` | CPU→GPU 전송을 uint8로 유지 → 대역폭 1/4 |
| 🎯 | **사전할당 원형 버퍼**(numpy 고정 크기) | `replay_buffer.py:13-19` | 동적 append/재할당 없음 |
| 🍀 | 흑백 + 84×84 리사이즈 + frame_skip=4 | `env_utils.py:16-26` | 표준 래퍼지만 입력 데이터량 크게 감소 |

### 속도 / 연산량
| | 내용 | 위치 | 효과 |
|---|---|---|---|
| 🎯 | `gamma_powers` **사전 계산** | `replay_buffer.py:27` | push마다 `gamma**i` 재계산 제거 |
| 🎯 | 행동선택 `torch.no_grad()` | `train.py:18` | 추론 시 그래프/메모리 절약 |
| 🎯 | 타깃 계산 `torch.no_grad()` | `train.py:30` | backward 대상에서 제외 |
| 🎯 | `train_freq=4` (4 env step당 1 학습) | `main.py:34` | 학습 연산 1/4 |
| 🎯 | PER **SumTree** O(log n) 샘플링 | `replay_buffer.py:108-118` | 선형탐색 대비 대폭 단축 |
| 🎯 | NoisyNet **factorized** 노이즈 | `model.py:41-52` | 노이즈 생성 O(n²)→O(n) |
| 🍀 | n-step 누적에 `deque(maxlen=n)` | `replay_buffer.py:25` | 윈도 관리 O(1) |

### 학습 안정화 (수렴/품질에 기여)
| | 내용 | 위치 | 효과 |
|---|---|---|---|
| 🎯 | grad norm **clipping 10.0** | `train.py:65` | PER 큰 TD-error에도 안정 |
| 🎯 | reward **sign clipping** {-1,0,+1} | `train.py:87` | 스케일 차이 흡수 (Atari 표준) |
| 🎯 | Adam **eps=1.5e-4** | `main.py:29` | Atari 권장값, 수치 안정 |
| 🎯 | PER **stratified** 구간 샘플링 | `replay_buffer.py:138-144` | 분산 감소 |
| 🎯 | IS weight **max 정규화** | `replay_buffer.py:170` | 업데이트 스케일 안정 |
| 🎯 | PER beta **anneal** 0.4→1.0 | `train.py:97` | 후반 편향 보정 강화 |
| 🍀 | `noop_max=30` 무작위 시작 | `env_utils.py:19` | 시작 상태 다양화(과적합 완화) |

> 참고: `main.py:27` 의 `target_net.eval()` 은 BN/Dropout이 없어 성능엔 무해하지만, Noisy와 함께 쓰면 타깃 노이즈를 끄는 **부작용**이 있음 → 6-2 항목에서 정합성 검토.

---
## 1. 리플레이 버퍼 메모리 최적화 (frame-stack 공유)

**문제** — `replay_buffer.py:13-14` 에서 `obs`와 `next_obs`를 각각 `(capacity, 4, 84, 84)` uint8로 통째 저장.
- 100,000 × 28,224 byte × **2 ≈ 5.6GB**.
- 프레임스택은 인접 transition끼리 3프레임이 겹치고, 어떤 transition의 `next_obs`는 다음 transition의 `obs`와 동일 → **중복**.

**개선 방향** — 단일 프레임만 원형 버퍼에 저장하고, 샘플 시점에 직전 4프레임을 모아 `(4,84,84)` 스택을 재구성.
- 메모리 약 **1/8**, `next_obs`는 `idx+1` 위치에서 즉석 생성.
- 에피소드 경계(done)에서 이전 에피소드 프레임이 섞이지 않도록 `episode_start`/`done` 마스크 처리 필요.

**효과** — `main.py:43` 의 `capacity=100_000` 을 논문 사양 **1_000_000** 으로 올릴 수 있어 학습 품질 직접 향상.

**작업량** — 중 (PER의 `_store_transition`/`_gather`도 단일 프레임 기준으로 함께 수정).

In [ ]:
# [참고용 스니펫 — 단일 프레임 저장 + 샘플 시 스택 재구성 개념]
# 실제 적용 시 ReplayBuffer 전반을 이 구조로 교체한다.
import numpy as np

class FrameStackBuffer:
    def __init__(self, capacity, frame_shape=(84, 84), stack=4):
        self.capacity, self.stack = capacity, stack
        # 프레임 한 장씩만 저장 -> 메모리 1/8
        self.frames  = np.zeros((capacity, *frame_shape), dtype=np.uint8)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.dones   = np.zeros(capacity, dtype=np.float32)
        self.ep_start = np.zeros(capacity, dtype=bool)  # 에피소드 시작 표시
        self.idx, self.size = 0, 0

    def _stack_at(self, i):
        # i를 마지막 프레임으로 하는 4-스택 구성 (에피소드 경계는 패딩)
        frames, valid = [], True
        for k in range(self.stack):
            j = (i - k) % self.capacity
            frames.append(self.frames[j])
            if k > 0 and self.ep_start[j]:   # 경계 만나면 그 이전은 복제로 패딩
                valid = False
            if not valid:
                pass
        return np.stack(frames[::-1], axis=0)  # (4,84,84)

# 핵심: obs/next_obs를 따로 저장하지 않고 인덱스 i와 (i+1)로 재구성한다.

---
## 2. Mixed Precision (AMP)

**문제** — `train.py:47-66` 의 forward+backward가 fp32 전부. Atari CNN은 AMP 이득이 큼.

**개선 방향** — `torch.cuda.amp.autocast` 로 forward/loss를 감싸고 `GradScaler` 로 backward.
- grad clip은 `scaler.unscale_()` 이후에 적용해야 정확.

**효과** — 보통 **1.3~2배** 속도, 메모리도 절감. **작업량** — 소.

In [ ]:
# [참고용 스니펫 — train_step 에 AMP 적용 형태]
import torch
import torch.nn.functional as F

scaler = torch.cuda.amp.GradScaler()   # 학습 시작 시 1회 생성

def train_step_amp(q_net, target_net, optimizer, buffer, batch_size,
                   gamma, use_double, use_noisy, beta, scaler):
    obs, actions, rewards, next_obs, dones, weights, indices = buffer.sample(batch_size, beta)
    optimizer.zero_grad(set_to_none=True)
    if use_noisy:
        q_net.reset_noise(); target_net.reset_noise()

    with torch.cuda.amp.autocast():
        elementwise_loss, td_errors = compute_loss(
            q_net, target_net, obs, actions, rewards, next_obs, dones, gamma, use_double)
        loss = (elementwise_loss * weights).mean()

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)                                   # clip 전에 unscale
    torch.nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
    scaler.step(optimizer)
    scaler.update()
    buffer.update_priorities(indices, td_errors.detach().cpu().numpy())
    return loss.item()

---
## 3. Distributional RL (C51) — 풀 Rainbow 완성

**문제** — 현재 스칼라 Q + Huber loss(`train.py:44`). Rainbow의 마지막 한 조각.

**개선 방향**
1. 네트워크 출력: `(batch, n_action, n_atoms)` (예: n_atoms=51), 행동축은 softmax로 확률분포.
2. support: `z = linspace(Vmin, Vmax, n_atoms)` (예: Vmin=-10, Vmax=10).
3. 타깃: Double DQN으로 행동 선택 → 분포를 `Tz = r + γ^n z` 로 이동/클램프 → **categorical projection** 으로 support에 재분배.
4. 손실: 타깃 분포 vs 예측 분포 **cross-entropy** (PER 우선순위는 이 CE 값/KL 사용).
5. Dueling/Noisy와 결합 시 마지막 FC 출력 차원을 `n_action * n_atoms` 로, Dueling은 atom 단위로 V/A 결합.

**효과** — 풀 Rainbow, 일반적으로 성능·안정성↑. **작업량** — 대 (model + loss + main 스위치 `USE_C51`).

In [ ]:
# [참고용 스니펫 — C51 핵심 (네트워크 출력 + projection)]
import torch, torch.nn.functional as F

N_ATOMS, VMIN, VMAX = 51, -10.0, 10.0

# 모델 forward 마지막: (batch, n_action*N_ATOMS) -> reshape -> softmax
def dist_forward(logits, n_action):
    logits = logits.view(-1, n_action, N_ATOMS)
    return F.softmax(logits, dim=2)          # 행동별 확률분포 p(s,a)

def project_target(next_dist, rewards, dones, gamma_n, device):
    z = torch.linspace(VMIN, VMAX, N_ATOMS, device=device)
    dz = (VMAX - VMIN) / (N_ATOMS - 1)
    Tz = (rewards.unsqueeze(1) + (1 - dones).unsqueeze(1) * gamma_n * z.unsqueeze(0))
    Tz = Tz.clamp(VMIN, VMAX)
    b  = (Tz - VMIN) / dz
    l, u = b.floor().long(), b.ceil().long()
    m = torch.zeros_like(next_dist)
    # 하한/상한 atom에 확률을 비율대로 분배 (인덱스 누적)
    m.scatter_add_(1, l, next_dist * (u.float() - b))
    m.scatter_add_(1, u, next_dist * (b - l.float()))
    return m                                   # 타깃 분포

# loss = -(target_dist * log(pred_dist[a])).sum(1)   # categorical cross-entropy

---
## 4. 벡터화 환경 (병렬 데이터 수집)

**문제** — `main.py:17` 단일 env. 수집 throughput이 단일 step에 묶임.

**개선 방향** — `gymnasium.vector.SyncVectorEnv`/`AsyncVectorEnv` 로 N개 env 동시 진행, 한 step에 N transition push.
- `train.py:train` 루프를 배치 step/배치 reset 기준으로 재작성 필요 (done 마스크 단위 reset).

**효과** — wall-clock 학습시간 단축. **작업량** — 대 (학습 루프 구조 변경).

In [ ]:
# [참고용 스니펫 — 벡터화 환경 골격]
import gymnasium as gym

def make_vec_env(env_id, num_envs, make_env_fn):
    return gym.vector.SyncVectorEnv([lambda: make_env_fn(env_id, seed=i)
                                     for i in range(num_envs)])

# 루프: obs (N,4,84,84) -> 한 번에 행동 N개 선택 -> step -> N개 transition push
# 자동 리셋(autoreset) 사용 시 done 처리/경계 프레임에 주의.

---
## 5. PER 샘플링/갱신 벡터화

**문제** — `replay_buffer.py:143-167`(sample), `179-182`(update)가 batch(32) 파이썬 for-loop + 요소별 `np.random.uniform`/트리 하강. 매 train step마다 반복되어 오버헤드.

**개선 방향**
- 구간 난수는 한 번에 벡터로 생성: `(np.arange(B) + rand(B)) * segment`.
- SumTree 하강은 numpy 배열 인덱싱으로 batch 동시 진행하거나, segment-tree 연산을 벡터화.
- `update_priorities` 도 numpy 벡터 연산으로.

**효과** — train step당 CPU 오버헤드↓ (특히 train_freq 잦을 때). **작업량** — 중.

In [ ]:
# [참고용 스니펫 — stratified 난수 벡터화]
import numpy as np

def sample_uniforms(total, batch_size):
    seg = total / batch_size
    # for-loop 없이 한 번에: 각 구간에서 균등 1개씩
    return (np.arange(batch_size) + np.random.random(batch_size)) * seg

# 트리 하강은 self.tree(numpy)에 대해 batch 인덱스 배열로 동시에 left/right 비교하여 진행.

---
## 6. NoisyNet 디바이스 + 기타 정리

**6-1. 노이즈 CPU 생성 후 복사** — `model.py:43,51-52` 의 `torch.randn(size)` 가 CPU에 생성된 뒤 GPU 버퍼로 copy. 매 train step × 여러 층 발생.
- 개선: `_scale_noise` 에서 파라미터와 같은 디바이스로 바로 생성 (`device=self.weight_mu.device`).

**6-2. target_net + Noisy 정합성** — `main.py:27` `target_net.eval()` 때문에 NoisyLinear가 노이즈를 무시(`model.py:59-62`), `train.py:61` 의 `target_net.reset_noise()` 가 사실상 무효. 원 Rainbow는 타깃망도 독립 노이즈 사용. (의도 확인 후 정합화)

**6-3. 데이터 전송** — `_gather`/`select_action`(`train.py:20`)의 CPU→GPU 전송에 pinned memory + `non_blocking=True` 적용 여지.

**효과** — 미세 속도 + 알고리즘 정합성. **작업량** — 소.

In [ ]:
# [참고용 스니펫 — 노이즈를 파라미터 디바이스에서 직접 생성]
import torch

def _scale_noise(self, size):
    x = torch.randn(size, device=self.weight_mu.device)  # CPU 생성 + copy 제거
    return x.sign().mul(x.abs().sqrt())

---
## 최종 체크리스트

- [ ] **1. 버퍼 메모리**: 단일 프레임 저장 + 스택 재구성, `capacity` 1M 상향 (ReplayBuffer/PER/main)
- [ ] **2. AMP**: `autocast` + `GradScaler`, clip 전 `unscale_` (train)
- [ ] **3. C51**: 분포 출력 + projection + CE loss + `USE_C51` 스위치 (model/train/main)
- [ ] **4. 벡터화 env**: `SyncVectorEnv`, 루프 배치화 (env_utils/train/main)
- [ ] **5. PER 벡터화**: stratified 난수/트리 하강/갱신 벡터화 (replay_buffer)
- [ ] **6. 정리**: 노이즈 디바이스, target noisy 정합성, non_blocking 전송 (model/train)

> 적용은 이 노트북과 별개로, 항목 단위로 반영 예정.